# Radiology Revenue Cycle — DDL
Creates all tables in `serverless_stable_jc9zgx_catalog.radiology_revenue_cycle` for the Two Reports demo.

**Tables**: patient, provider, service_location, payer, encounter, claim_header, claim_line, denial, payment, ar_aging_snapshot, authorization

**Trust Traps Planted (8 traps across 6 dimensions)**:
| Dimension | Table | Defect |
|-----------|-------|--------|
| Completeness | claim_header, claim_line | ~8% NULL allowed_amount on paid claims |
| Validity | denial | Invalid CARC codes (CO-999, PR-ZZZ, OA-500, PI-300) |
| Consistency | claim_header | Payer name variants vs canonical payer table |
| Uniqueness | claim_header | Rebill duplicates with unreliable is_rebill flag |
| Timeliness | ar_aging_snapshot | System A 9 days stale vs System B nightly |
| Accuracy (A) | claim_line | ~6% TC/26 component overlap with global charges |
| Accuracy (B) | claim_line | PAY-011/PAY-012 allowed_amount = charge_amount (clearinghouse error) |
| Accuracy (C) | claim_line | PAY-001 Q1 2026 stale contracted rates (12% below actual payment) |

In [0]:
%sql
USE CATALOG serverless_stable_jc9zgx_catalog;
CREATE SCHEMA IF NOT EXISTS radiology_revenue_cycle
COMMENT 'Pinnacle Radiology Partners — Revenue Cycle Management data for the Two Reports demo. Contains intentionally seeded DQ defects across all six quality dimensions.';
USE SCHEMA radiology_revenue_cycle;

## Reference / Dimension Tables

In [0]:
%sql
CREATE OR REPLACE TABLE patient (
  patient_id        STRING    NOT NULL  COMMENT 'Primary key — synthetic patient identifier',
  mrn               STRING    NOT NULL  COMMENT 'Medical Record Number',
  first_name        STRING              COMMENT 'Patient first name',
  last_name         STRING              COMMENT 'Patient last name',
  dob               DATE                COMMENT 'Date of birth',
  sex               STRING              COMMENT 'M|F|O',
  zip_code          STRING              COMMENT '5-digit ZIP code',
  insurance_id      STRING              COMMENT 'Insurance member ID',
  primary_payer_id  STRING              COMMENT 'FK to payer.payer_id',
  created_at        TIMESTAMP           COMMENT 'Record creation timestamp'
)
COMMENT 'Patient demographics for Pinnacle Radiology Partners. ~5,000 patients across 12 locations.';

In [0]:
%sql
CREATE OR REPLACE TABLE provider (
  provider_npi        STRING    NOT NULL  COMMENT 'Primary key — National Provider Identifier (10-digit)',
  provider_name       STRING    NOT NULL  COMMENT 'Full name (Last, First Credential)',
  specialty           STRING              COMMENT 'Radiology subspecialty (e.g., neuroradiology, interventional, breast imaging)',
  credential          STRING              COMMENT 'MD|DO',
  group_affiliation   STRING              COMMENT 'Practice group within Pinnacle',
  location_id         STRING              COMMENT 'Primary location FK to service_location.location_id',
  is_active           BOOLEAN             COMMENT 'Active status'
)
COMMENT 'Radiologists and referring physicians. ~50 providers.';

In [0]:
%sql
CREATE OR REPLACE TABLE service_location (
  location_id     STRING    NOT NULL  COMMENT 'Primary key — location identifier',
  location_name   STRING    NOT NULL  COMMENT 'Facility display name',
  location_type   STRING    NOT NULL  COMMENT 'freestanding_imaging|hospital_based|mobile',
  address_line1   STRING              COMMENT 'Street address',
  city            STRING              COMMENT 'City',
  state           STRING              COMMENT 'Two-letter state abbreviation',
  zip_code        STRING              COMMENT '5-digit ZIP code',
  pos_code        STRING    NOT NULL  COMMENT 'CMS Place of Service code (e.g., 22=outpatient hospital, 11=office, 15=mobile)',
  phone           STRING              COMMENT 'Main phone number',
  is_active       BOOLEAN             COMMENT 'Active status'
)
COMMENT '12 service locations: 8 freestanding imaging centers, 3 hospital-based reading groups, 1 mobile MRI unit.';

In [0]:
%sql
CREATE OR REPLACE TABLE payer (
  payer_id            STRING    NOT NULL  COMMENT 'Primary key — payer identifier',
  payer_name          STRING    NOT NULL  COMMENT 'Canonical/authoritative payer name — the SINGLE SOURCE OF TRUTH for payer identity',
  payer_category      STRING    NOT NULL  COMMENT 'commercial|medicare|medicaid|tricare|workers_comp|self_pay',
  contract_type       STRING              COMMENT 'fee_for_service|capitated|bundled|value_based',
  timely_filing_days  INT                 COMMENT 'Days allowed for claim submission per contract',
  is_active           BOOLEAN             COMMENT 'Active status'
)
COMMENT 'Canonical payer reference table. 12 contracted payers. This is the authoritative source for payer identity — claim_header.payer_name is denormalized and inconsistent.';

## Transactional / Fact Tables

In [0]:
%sql
CREATE OR REPLACE TABLE encounter (
  encounter_id            STRING    NOT NULL  COMMENT 'Primary key — unique encounter/study identifier',
  patient_id              STRING    NOT NULL  COMMENT 'FK to patient.patient_id',
  provider_npi            STRING    NOT NULL  COMMENT 'FK to provider.provider_npi — reading radiologist',
  location_id             STRING    NOT NULL  COMMENT 'FK to service_location.location_id',
  study_date              DATE      NOT NULL  COMMENT 'Date the imaging study was performed',
  modality                STRING    NOT NULL  COMMENT 'MRI|CT|XR|US|MG|NM|IR',
  accession_number        STRING              COMMENT 'PACS accession number for the study',
  order_status            STRING    NOT NULL  COMMENT 'completed|cancelled|no_show|in_progress',
  referring_npi           STRING              COMMENT 'NPI of the referring/ordering physician',
  referring_provider_name STRING              COMMENT 'Name of referring physician',
  clinical_indication     STRING              COMMENT 'Clinical reason for exam (free text from order)',
  created_at              TIMESTAMP           COMMENT 'Record creation timestamp'
)
COMMENT 'Imaging encounters/studies. ~40,000 rows representing 12 months of radiology volume across all modalities.';

In [0]:
%sql
CREATE OR REPLACE TABLE claim_header (
  claim_id              STRING    NOT NULL  COMMENT 'Primary key — unique claim identifier',
  encounter_id          STRING    NOT NULL  COMMENT 'FK to encounter.encounter_id',
  patient_id            STRING    NOT NULL  COMMENT 'FK to patient.patient_id',
  payer_id              STRING    NOT NULL  COMMENT 'FK to payer.payer_id',
  payer_name            STRING              COMMENT 'Denormalized from source system — WARNING: inconsistent with payer.payer_name. Contains variants like BCBS/Blue Cross BS/BlueCross BlueShield. This is the CONSISTENCY trap.',
  rendering_npi         STRING    NOT NULL  COMMENT 'NPI of the rendering provider',
  billing_npi           STRING    NOT NULL  COMMENT 'NPI of the billing entity',
  location_id           STRING    NOT NULL  COMMENT 'FK to service_location.location_id',
  claim_type            STRING    NOT NULL  COMMENT 'professional|facility',
  filing_date           DATE      NOT NULL  COMMENT 'Date claim was submitted to payer',
  total_charge_amount   DOUBLE    NOT NULL  COMMENT 'Sum of all line charges',
  total_allowed_amount  DOUBLE              COMMENT 'Sum of allowed amounts — NULL for ~8% of paid claims where ERA posted without allowed. This is the COMPLETENESS trap.',
  total_paid_amount     DOUBLE              COMMENT 'Sum of payments received',
  claim_status          STRING    NOT NULL  COMMENT 'paid|denied|pending|partial|void',
  adjudication_date     DATE                COMMENT 'Date payer adjudicated the claim',
  is_rebill             BOOLEAN             COMMENT 'True if this is a corrected/re-filed claim. Only reliably populated after 2025-03-01 — the UNIQUENESS trap.',
  original_claim_id     STRING              COMMENT 'Self-referential FK to the original claim_id when is_rebill=true. NULL for original claims and pre-March 2025 rebills.',
  created_at            TIMESTAMP           COMMENT 'Record creation timestamp'
)
COMMENT 'Claim headers for all filed claims. ~38,000 rows. Contains three trust traps: COMPLETENESS (null allowed_amount), CONSISTENCY (payer_name variants), UNIQUENESS (rebill duplicates).'
TBLPROPERTIES (
  'dq.trap.completeness' = '~8% of paid claims have NULL total_allowed_amount because ERA posted without allowed amounts',
  'dq.trap.consistency' = 'payer_name is free-text and inconsistent with canonical payer.payer_name (e.g. BCBS vs Blue Cross Blue Shield)',
  'dq.trap.uniqueness' = '~3% of claims are rebill duplicates where is_rebill flag was not populated before March 2025'
);

In [0]:
%sql
CREATE OR REPLACE TABLE claim_line (
  claim_line_id           STRING    NOT NULL  COMMENT 'Primary key — unique line identifier',
  claim_id                STRING    NOT NULL  COMMENT 'FK to claim_header.claim_id',
  line_number             INT       NOT NULL  COMMENT 'Line sequence number within the claim',
  cpt_code                STRING    NOT NULL  COMMENT 'CPT/HCPCS procedure code',
  modifier_1              STRING              COMMENT 'First modifier (e.g., 26=professional component, TC=technical component, NULL=global)',
  modifier_2              STRING              COMMENT 'Second modifier',
  icd10_code              STRING    NOT NULL  COMMENT 'Primary diagnosis code for this line',
  units                   INT       NOT NULL  COMMENT 'Number of units billed',
  charge_amount           DOUBLE    NOT NULL  COMMENT 'Billed charge for this line',
  allowed_amount          DOUBLE              COMMENT 'Payer-allowed amount. ACCURACY TRAPS: (A) ~6% of claims have overlapping global+component lines for same CPT; (B) PAY-011/PAY-012 have allowed=charge due to clearinghouse mapping error; (C) PAY-001 Q1 2026 carries stale rate 12% below actual payment. COMPLETENESS TRAP: NULL for ~8% of paid lines.',
  paid_amount             DOUBLE              COMMENT 'Amount paid by payer for this line',
  adjustment_amount       DOUBLE              COMMENT 'Adjustment amount (negative = reduces revenue)',
  adjustment_reason_code  STRING              COMMENT 'CARC code',
  remark_code             STRING              COMMENT 'RARC code — additional explanation from payer',
  place_of_service        STRING    NOT NULL  COMMENT 'CMS Place of Service code',
  service_date            DATE      NOT NULL  COMMENT 'Date service was rendered'
)
COMMENT 'Claim line detail. ~95,000+ rows (avg 2.5 lines per claim plus overlap lines). Contains 4 traps: COMPLETENESS (null allowed), ACCURACY-A (TC/26 overlap with global), ACCURACY-B (allowed=billed for 2 payers), ACCURACY-C (stale BCBS fee schedule Q1 2026).'
TBLPROPERTIES (
  'dq.trap.completeness' = '~8% of paid lines have NULL allowed_amount',
  'dq.trap.accuracy_a' = '~6% of claims have global line (no modifier) PLUS duplicate TC and 26 lines for same CPT, triple-counting revenue',
  'dq.trap.accuracy_b' = 'PAY-011 and PAY-012 have allowed_amount = charge_amount due to clearinghouse 835 mapping error',
  'dq.trap.accuracy_c' = 'PAY-001 (BCBS) Q1 2026 claims carry stale contracted rate in allowed_amount, 12% below actual payment amount'
);

In [0]:
%sql
CREATE OR REPLACE TABLE denial (
  denial_id         STRING    NOT NULL  COMMENT 'Primary key — unique denial identifier',
  claim_id          STRING    NOT NULL  COMMENT 'FK to claim_header.claim_id',
  claim_line_id     STRING              COMMENT 'FK to claim_line.claim_line_id — NULL indicates a header-level denial (entire claim rejected)',
  denial_date       DATE      NOT NULL  COMMENT 'Date the denial was issued by the payer',
  carc_code         STRING    NOT NULL  COMMENT 'Claim Adjustment Reason Code — includes INVALID codes CO-999, PR-ZZZ, OA-500, PI-300 from legacy migration. This is the VALIDITY trap.',
  rarc_code         STRING              COMMENT 'Remittance Advice Remark Code',
  denial_category   STRING              COMMENT 'auth|coding|medical_necessity|eligibility|duplicate|timely_filing|unclassified',
  denied_amount     DOUBLE    NOT NULL  COMMENT 'Dollar amount denied',
  appeal_status     STRING              COMMENT 'none|submitted|in_review|overturned|upheld',
  appeal_date       DATE                COMMENT 'Date appeal was submitted',
  overturn_date     DATE                COMMENT 'Date denial was overturned (if applicable)',
  overturn_amount   DOUBLE              COMMENT 'Amount recovered through appeal'
)
COMMENT 'Claim denials with CARC/RARC codes and appeal workflow. ~4,500 rows. Contains the VALIDITY trap: ~7% of denials use CARC codes that do not exist in the X12 835 standard.'
TBLPROPERTIES (
  'dq.trap.validity' = 'Invalid CARC codes planted: CO-999, PR-ZZZ, OA-500, PI-300 — entered by legacy billing system during migration, never validated against X12 code set'
);

In [0]:
%sql
CREATE OR REPLACE TABLE payment (
  payment_id        STRING    NOT NULL  COMMENT 'Primary key — unique payment transaction identifier',
  claim_id          STRING    NOT NULL  COMMENT 'FK to claim_header.claim_id',
  claim_line_id     STRING    NOT NULL  COMMENT 'FK to claim_line.claim_line_id',
  payer_id          STRING    NOT NULL  COMMENT 'FK to payer.payer_id — may differ from claim payer_id in crossover scenarios',
  payment_date      DATE      NOT NULL  COMMENT 'Date payment was received/posted',
  payment_amount    DOUBLE    NOT NULL  COMMENT 'Amount paid in this transaction',
  adjustment_amount DOUBLE              COMMENT 'Adjustment posted with this payment',
  adjustment_code   STRING              COMMENT 'Reason code for adjustment',
  check_number      STRING              COMMENT 'Check or EFT reference number',
  era_id            STRING              COMMENT 'Electronic Remittance Advice identifier — links to the 835 file'
)
COMMENT 'Payment/remittance transactions. ~85,000 rows. Multiple payments per claim possible (primary, secondary, patient). Proves claims were paid even when allowed_amount is NULL on claim_line.';

In [0]:
%sql
CREATE OR REPLACE TABLE ar_aging_snapshot (
  snapshot_date       DATE      NOT NULL  COMMENT 'Date this snapshot was captured',
  payer_id            STRING    NOT NULL  COMMENT 'FK to payer.payer_id',
  location_id         STRING    NOT NULL  COMMENT 'FK to service_location.location_id',
  aging_bucket        STRING    NOT NULL  COMMENT 'current|31_60|61_90|91_120|121_plus',
  claim_count         INT       NOT NULL  COMMENT 'Number of outstanding claims in this bucket',
  outstanding_amount  DOUBLE    NOT NULL  COMMENT 'Total outstanding AR dollars in this bucket',
  snapshot_source     STRING    NOT NULL  COMMENT 'system_a = hospital-based locations (STALE by 9 days), system_b = freestanding centers (nightly refresh). This is the TIMELINESS trap.'
)
COMMENT 'Periodic AR aging snapshots by payer, location, and bucket. ~8,640 rows. Contains the TIMELINESS trap: system_a data is 9 days stale while system_b refreshes nightly.'
TBLPROPERTIES (
  'dq.trap.timeliness' = 'system_a (hospital-based) last refreshed 2026-05-06 (9 days stale); system_b (freestanding) refreshes nightly. Reports that filter on snapshot_date=CURRENT_DATE silently exclude all hospital locations.'
);

In [0]:
%sql
CREATE OR REPLACE TABLE authorization (
  auth_id           STRING    NOT NULL  COMMENT 'Primary key — unique authorization identifier',
  patient_id        STRING    NOT NULL  COMMENT 'FK to patient.patient_id',
  payer_id          STRING    NOT NULL  COMMENT 'FK to payer.payer_id',
  cpt_code          STRING    NOT NULL  COMMENT 'CPT code the authorization covers',
  auth_number       STRING    NOT NULL  COMMENT 'Authorization/reference number from payer',
  auth_status       STRING    NOT NULL  COMMENT 'approved|denied|pending|expired',
  requested_date    DATE      NOT NULL  COMMENT 'Date auth was requested',
  approved_date     DATE                COMMENT 'Date auth was approved (NULL if denied/pending)',
  expiration_date   DATE                COMMENT 'Date auth expires',
  authorized_units  INT                 COMMENT 'Number of units/visits authorized'
)
COMMENT 'Prior authorizations for advanced imaging. ~15,000 rows. No hard FK to claim_header — linkage is via soft match on (patient_id, cpt_code, date range). This gap causes CO-197 denials when auth exists but is not linked at submission.';

## Verification

In [0]:
%sql
SHOW TABLES IN serverless_stable_jc9zgx_catalog.radiology_revenue_cycle;